# **Initialization**

In [ ]:
# Essential Libraries
import os
from os import path
import glob
from pathlib import Path
import datetime
import math
import warnings

# Data Handling
import numpy as np
import pandas as pd
import random
import sklearn

# Statistics
import scipy
from statistics import mean, stdev

# Special Libraries
#import matlab.engine

# Microscopy File Handling
import tifffile
from tifffile import TiffWriter
import nd2
import skimage as ski
from skimage import io

# Data Handling/Multiprocessing
import multiprocessing
from multiprocessing import Pool
import trackpy as tp

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Miscellaneous
import itertools
from itertools import chain
import alive_progress
from alive_progress import alive_bar
import tqdm

# Open XML Files
import xml.etree.ElementTree as ET

# AnDi Challenge Package
import andi_datasets

In [ ]:
global_filepath = Path(r'E:\2026\ExoIII\20260120 - Chimera RNase A Test_JH-ELN-0002-0014\MOSAIC Outputs\\')

# **Utilities**

In [ ]:
def generate_output_filepath(parent = global_filepath, file_string = None, subfolder=None):
    """
    Generates output filepath and subfolder (if applicable) to save files.

    Parameters: 
    -   parent: pathlib.Path
            Parent filepath used to save folder. Often is the main filepath that is used to open files (global_filepath).

    -   file_string: str or pathlib.Path
            Stem of file name. Must contain file extension.

    -   subfolder: str or pathlib.Path
            Subdirectory used to save file separately from parent. Convenient if the parent directory is global_filepath 
            and a subdirectory will be used to store data from a specific run or a specific type of processed data.

    Returns: 
    -   output_filepath: pathlib.Path
            Path object that can be used to save files (e.g. in pd.to_csv() function)
    """

    # Checks whether file_string was added
    if not file_string: 
        raise ValueError('Must include filename stem and extension')
    
    # Converts file_string to Path object if required
    if not isinstance(file_string, Path):
        file_string = Path(file_string)

    # Checks that file_string has extension
    if not file_string.suffix:
        raise ValueError('File path must include extension')
    
    # Adds subfolder to output path if specified
    if subfolder:
        subfolder_path = parent / Path(subfolder)

        # Ensures that subfolder exists
        if not subfolder_path.exists():
            subfolder_path.mkdir(parents = True, exist_ok = True)

        # Generates output filepath in output subfolder
        output_filepath = subfolder_path / file_string

    else:
        # Generates output filepath in parent folder
        output_filepath = parent / file_string
    
    return output_filepath

In [ ]:
def threshold_SPT(df):
    """
    Sets origin of each trajectory to (0,0) to make calculations and plotting simpler. 
    Note: This function exists entirely in Pandas and uses the pd.concat() function, which has O(N**2) time complexity (https://stackoverflow.com/questions/36489576/why-does-concatenation-of-dataframes-get-exponentially-slower). 
    This is VERY slow in cases where more than 100 trajectories must be handled at a time, but it has the added advantage of high stability regardless of the number of columns.

    Parameters:
    -   df: DataFrame
            Contains trajectory data from MOSAIC. Must minimally contain columns for Trajectory, Frame, x, and y

    Returns:
    -   thresholded_df: DataFrame
            Copy of df with columns for relative distance in x and y added. 
    
    """
    # Generates empty DataFrame to contain outputs of thresholding function
    thresholded_df = pd.DataFrame()
    
    # Iterates through input DataFrame by trajectory
    for _, traj_df in df.groupby('Trajectory'):

        # Determines initial values of x and y
        x_0 = traj_df['x'].iloc[0]
        y_0 = traj_df['y'].iloc[0]

        # Subtracts initial values of x and y from corresponding coordinate columns
        rel_x = traj_df['x'] - x_0
        rel_y = traj_df['y'] - y_0

        # Adds result of subtraction to traj_df
        traj_df['x (rel)'] = rel_x
        traj_df['y (rel)'] = rel_y

        # Concatenates thresholded coordinate columns to thresholded_df
        thresholded_df = pd.concat([thresholded_df, traj_df], ignore_index=True)
    
    return thresholded_df

In [ ]:
def moving_average(y, sampled_pts=5):
    """
    Calculates the moving average of a 1D array. Uses reflection at boundary to produce moving average with the same length as the original time series,
    as it is assumed that the change in value in the first and last couple of points is not significant. A different boundary operation may be required 
    for data for which this assumption is inappropriate. 
    
    Parameters: 
    -   y: list or ndarray
            Contains time-series values to be averaged.
    -   sampled_pts: int
            Width of window used for averaging. Must be both positive and odd.
            
    Returns: 
    -   mv_avg_y: ndarray
            Time series with moving average applied.
    """
    y = np.asarray(y, dtype=float)
    pad = sampled_pts // 2

    # Reflect-pad at the edges to avoid edge distortion
    y_padded = np.pad(y, pad_width=pad, mode='reflect')

    kernel = np.ones(sampled_pts) / sampled_pts
    mv_avg_y = np.convolve(y_padded, kernel, mode='valid')

    return mv_avg_y

In [ ]:
def plot_1D(traj, real_time = False, frame_rate = 60, averaging = False, averaging_window = 5):
    """
    Plots each dimension of a multi-dimensional time series as a 1D trace. Time series can be expressed in frames or in real time.
    Displays plot directly from function call. 

    Parameters: 
    -   traj: ndarray (T, D+1)
            Must contain the time series frame index in first column and the series values in subsequent columns.

    -   real_time: bool
            Refers to real values on x axis if True, otherwise plots with respect to frame
    
    -   frame_rate: int or float
            Relates the frame count in the time series to real-time values. Not called if real_time is False. 
    
    -   averaging: bool
            Plots moving average of each time series if True.

    -   averaging_window: int
            Window of moving average that is used if averaging is True. Not called if averaging is False.

    Internal Dependencies: 
    -   moving_average

    Returns: 
    -   None
    """
    # traj should be numpy array of shape (T, D+1). First column is frame.
    plt.figure(figsize=(24,12))
    
    t = traj[:,0]
    if real_time:
        t = t / frame_rate
        plt.xlabel('Time (s)', fontsize=36)
    else: 
        plt.xlabel('Frame', fontsize=36)
    
    num_dimensions = traj.shape[1]

    for i in np.arange(start = 1, stop = num_dimensions, step = 1):
        y = traj[:,i]
        plt.plot(t, y)

        if averaging:
            y_avg = moving_average(y, sampled_pts=averaging_window)
            plt.plot(t, y_avg, color='black')

    plt.ylabel('Distance (μm)', fontsize=36)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_2D(traj, real_time = False, frame_rate = 60):
    """
    Creates a 2D plot of the trajectory. Note that this plot uses a rainbow colormap to note the time at which the particle traverses a point, 
    so it may lead to artifiical grouping by viewers in an optical illusion. This plot should not be used to infer a stepwise behavior. 
    Function creates plot when called.

    Parameters: 
    -   traj: ndarray (T, D+1)
            Encodes position information for coordinates with respect to time (first column).

    -   real_time: bool
            Colorbar ticks are real time stamps if True, frames if False.

    -   frame_rate: int or float
            Rate of acquisitions (per second) used to convert between frames and time stamps. Not called if real_time is False.

    Returns: 
    -   None
    """
    from matplotlib.collections import LineCollection
    fig, ax = plt.subplots(figsize=(12,10))
    
    points = traj[:, 1:3]
    segments = np.stack([points[:-1], points[1:]], axis=1)

    t = traj[:,0]
    if real_time:
        t = t / frame_rate
        time_label = 'Time (s)'
    else: 
        time_label = 'Frame'
    
    time_points = np.linspace(0, t[-1], len(t))

    lc = LineCollection(segments, cmap='jet', norm=plt.Normalize(0, t[-1]), alpha=1)

    lc.set_array(time_points[:len(segments)])
    lc.set_linewidth(1)
    ax.add_collection(lc)
    
    plt.axis('equal')
    plt.xlabel('X Coordinate (μm)')
    plt.ylabel('Y Coordinate (μm)')
    
    plt.colorbar(lc, ax=ax, label=time_label)
    plt.show()


In [ ]:
def winnow_trajectories_by_frame_count(df, num_frames = 361):
    """
    Only retains trajectories that have the specified number of frames. Note: Uses pd.concat() function, which has O(n**2) time complexity.
    This function should be modified in future implementations. 

    Parameters: 
    -   df: DataFrame
            DataFrame that contains trajectory information (typically started as MOSAIC outputs). Must minimally contain a Trajectory column.
    
    -   num_frames: int
            Number of frames that will be used to filter list of trajectories. Must be a natural number.
    
    Returns: 
    -   retained_trajectories: DataFrame
            DataFrame that contains trajectories with number of frames equal to num_frames. 
    """
    retained_trajectories = pd.DataFrame()
    for _, traj_df in df.groupby('Trajectory'):
        if len(traj_df) == num_frames:
            retained_trajectories = pd.concat([retained_trajectories, traj_df])

    return retained_trajectories

def select_random_trajectories(df, num_trajectories = 500):
    """
    Selects random trajectories for analysis. Will return all trajectories if fewer trajectories are avaialble than are requested.
    Note: Uses pd.concat() function, which has O(n**2) time complexity. This function should be modified in future implementations. 
    Parameters: 
    -   df: DataFrame
            DataFrame containing all trajectories. Must minimally contain a Trajectory column.
    
    -   num_trajectories: int
            Number of requested trajectories. Must be a natural number
            
    Returns: 
    -   output: DataFrame
            Contains randomly selected trajectories.
    """
    trajectories = list(df['Trajectory'].unique())
    trajectories = [int(item) for item in trajectories]
    print(f'{len(trajectories)} trajectories meet the filtering criterion')
    num_trajectories = min(len(trajectories), num_trajectories)
    trajectories = random.sample(trajectories, num_trajectories)
    output = pd.DataFrame()
    for traj, traj_df in df.groupby('Trajectory'):
        if traj in trajectories:
            output = pd.concat([output, traj_df])

    return output


# **Miscellaneous**

In [ ]:
compiled_df = pd.read_csv(global_filepath / Path('Simulated_Dataset_Theoretical_Fingerprint_GLOBAL.csv'))
trajectory_df = pd.read_csv(rf'C:\Users\joshu\OneDrive\Graduate School\Research\ExoIII\Benchmarking\Trajectory Simulations\Simulated_Dataset_Theoretical.csv')
print(trajectory_df)

trajectory_states = []

for id, id_df in trajectory_df.groupby('ID'):
    models = id_df['Model'].unique()
    trajectory_states.append([id, ','.join(models)])

trajectory_states_df = pd.DataFrame(trajectory_states, columns=['ID', 'States'])

compiled_df['Model'] = trajectory_states_df['States']

for model, model_df in tqdm.tqdm(compiled_df.groupby('Model'), desc = 'Saving single-model files'):
    output_path = generate_output_filepath(file_string = f'Simulated_Trajectories_Theoretical_{model}_Fingerprint_GLOBAL.csv')
    model_df.drop(labels = ['Model'], inplace = True, axis = 1)
    model_df.to_csv(output_path, index = False)

# **Open and Preprocess MOSAIC Files**

In [ ]:
files = global_filepath.glob('*.csv')

file_count = 0
data = pd.DataFrame()
for file in files: 
    df = pd.read_csv(file)
    df['Trajectory'] = df['Trajectory'] + file_count
    df['Dataset'] = file.stem
    data = pd.concat([data, df],ignore_index=True)
    file_count += 10000

print(data['Trajectory'].nunique())    

data['x'] = data['x'] * 0.55
data['y'] = data['y'] * 0.55

In [ ]:
thresholded_data = threshold_SPT(data)
thresholded_data['Time (s)'] = thresholded_data['Frame'] / 12
print(thresholded_data)

# **Evaluate Localization Precision**

In [ ]:
def construct_pdist(data, lbound=-150, ubound=150, n_bins=60, set_density=False):
    """
    Standard function for constructing probability distribution function using np.histogram function. Returns information that can be used to plot more accurately.
    
    Parameters: 
    -   data: list, ndarray, pandas.Series, etc.
            Self-explanatory. One dimension.

    -   lbound: int or float
            Lower limit used for calculation.

    -   ubound: int or float
            Upper limit used for calculation.

    -   n_bins: int
            Number of bins used for histogram calculation. Must be a natural number.

    -   set_density: bool
            Determines whether the histogram is a probability density. generally False since probabilities are computed in next line.

    Returns: 
    -   counts: np.ndarray
            Array of values corresponding to observations within a specific bin
    
    -   bin_centers: np.ndarray
            Centers of bins

    -   probs: np.ndarray
            Probability values for all bins
    
    -   bin_widths: np.ndarray
            Widths of all bins (used for plotting)
    """
    counts, bin_edges = np.histogram(data, bins = np.linspace(lbound,ubound,n_bins), density=set_density)
    probs = counts / np.sum(counts)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_widths = np.diff(bin_edges)
    return counts, bin_centers, probs, bin_widths

def gaussian(x, amp, mean, stdev):
    """
    Fits a Gaussian distribution. CANNOT DECOUPLE STANDARD DEVIATION FROM AMPLITUDE.
    
    Parameters: 
    -   x: np.ndarray or list
            independent variable coordinates for fit.

    Fit variables: 
    -   amp: float
            Amplitude of Gaussian fit.
    
    -   mean: float
            Mean of Gaussian fit.

    -   stdev: float
            Standard deviation of Gaussian fit.
            
    Returns: 
    -   Gaussian fit (Used with scipy.optimize.curve_fit. Not useful alone)
    """

    return amp * np.exp(-((x - mean) ** 2) / (2 * stdev ** 2))

def gaussian_fit(data, lbound=-500, ubound=500, n_bins=501, set_density=True):
    """
    Wraps gaussian fitting function to produce probability density function with accompanying Gaussian fitting parameters.
    
    Parameters: 
    -   data: list, np.ndarray, or pandas.Series
            One-dimensional data structure.
    
    -   lbound: int or float
            Lower limit used to calculate histogram bins.
    
    -   ubound: int or float
            Upper limit used to calculate histogram bins.
            
    -   n_bins: int
            Number of bins that will be used to generate histogram. Must be natural number.
            
    -   set_density: bool
            Histogram is probability density function if True, otherwise is count-based. 
            
    Internal Dependencies: 
    -   gaussian: function used for curve fitting.
    
    Returns: 
    -   popt: list
            Fit parameters for Gaussian distribution. [amp, mean, standard deviation]
            
    x_fit: np.ndarray
            Array of values used for fit of independent variable.
            
    y_fit: np.ndarray
            Array of dependent values generated with Gaussian fit function from independent variable.
    """
    
    # Create lists of x and y values for Gaussian fit
    bin_heights, bin_edges = np.histogram(data, bins=np.linspace(lbound, ubound, n_bins), density=set_density)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Initial guess for parameters
    p0 = [np.max(bin_heights), np.mean(data), np.std(data)]

    # Calculate Gaussian fit
    popt, _ = scipy.optimize.curve_fit(gaussian, bin_centers, bin_heights, p0=p0)

    # Generate smooth curve for plotting
    x_fit = np.linspace(min(bin_edges), max(bin_edges), 300)
    y_fit = gaussian(x_fit, *popt)

    return popt, x_fit, y_fit

In [ ]:
sigmas = []

# Generate figure and calculate Gaussian fit SD for each trajectory
for traj, traj_df in thresholded_data.groupby('Trajectory'):
    print(f'Localization Image for Trajectory {traj}.')
    # Create gridspec figure for plotting
    fig = plt.figure(figsize=(30, 9))
    #plt.suptitle(f'Localization Precision Calculation for Trajectory {traj}')
    gs = gridspec.GridSpec(nrows = 10, ncols = 10, figure=fig)
    gs.update(wspace=0.1, hspace=0.3)

    ax0  = fig.add_subplot(gs[0:11, 0:4])  # Scatter plot
    ax1x = fig.add_subplot(gs[0:5, 5:8])  # x coordinate over time
    ax1y = fig.add_subplot(gs[6:11, 5:8])  # y coordinate over time
    ax2x = fig.add_subplot(gs[0:5, 8:11])    # x coordinate probability distribution and fit
    ax2y = fig.add_subplot(gs[6:11, 8:11])    # y coordinate probability distribution and fit

    # Plot graphs
    ax0.scatter(traj_df['x (rel)'], traj_df['y (rel)'], s=4, alpha=1)
    ax0.set_xlabel('x (nm)', fontsize=12)
    ax0.set_ylabel('y (nm)', fontsize=12)
    ax0.set_xlim((-100, 100))
    ax0.set_ylim((-100, 100))

    ax1x.plot(traj_df['Time (s)'], traj_df['x (rel)'], color='gray', linewidth=0.5)
    ax1x.set_ylabel('x (nm)', fontsize=12)
    ax1x.set_xlabel('Frame', fontsize=12)
    ax1x.set_ylim((-100, 100))

    ax1y.plot(traj_df['Time (s)'], traj_df['y (rel)'], color='gray', linewidth=0.5)
    ax1y.set_ylabel('y (nm)', fontsize=12)
    ax1y.set_xlabel('Frame', fontsize=12)
    ax1y.set_ylim((-100, 100))

    ax2x.hist(traj_df['x (rel)'], bins=56, density=True, orientation='horizontal')
    ax2x.set_xlabel('P(d)')
    ax2x.set_ylim((-100,100))
    ax2x.tick_params(left=False, bottom=True, labelleft=False, labelbottom=True)

    ax2y.hist(traj_df['y (rel)'], bins=56, density=True, orientation='horizontal')
    ax2y.set_xlabel('P(d)')
    ax2y.set_ylim((-100,100))
    ax2y.tick_params(left=False, bottom=True, labelleft=False, labelbottom=True)
    
 # Compute and overlay Gaussian fit
    try:
        (x_amp, x_mu, x_sigma), x_fit, x_fit_y = gaussian_fit(traj_df['x (rel)'])
        (y_amp, y_mu, y_sigma), y_fit, y_fit_y = gaussian_fit(traj_df['y (rel)'])
        
        ax2x.plot(x_fit_y, x_fit, 'r-', lw=1.5, label=f"σ={x_sigma:.2f}")
        ax2y.plot(y_fit_y, y_fit, 'r-', lw=1.5, label=f"σ={y_sigma:.2f}")

        ax2x.legend(fontsize=12)
        ax2y.legend(fontsize=12)

        sigmas.append([traj, x_sigma, y_sigma])
    except Exception as e:
        print(f"Gaussian fit failed for trajectory {traj}: {e}")

    plt.tight_layout()
    plt.show()

In [ ]:
for sublist in sigmas:
    sublist.append(np.sqrt(sublist[1]**2+sublist[2]**2))

sigmas_list_df = pd.DataFrame(sigmas, columns=['Trajectory', 'sigma x', 'sigma y', 'sigma hypotenuse'])

sigmas_list_df.to_csv(rf'{global_filepath}/Localization_Precision.csv')

x_mean, x_std = np.mean(sigmas_list_df['sigma x']), np.std(sigmas_list_df['sigma x'])
y_mean, y_std = np.mean(sigmas_list_df['sigma y']), np.std(sigmas_list_df['sigma y'])
gen_mean, gen_std = np.mean(sigmas_list_df['sigma hypotenuse']), np.std(sigmas_list_df['sigma hypotenuse'])

print(f'The localization precision in the x coordinate is {x_mean} -/+ {x_std}.')
print(f'The localization precision in the y coordinate is {y_mean} -/+ {y_std}.')
print(f'The two-dimensional localization precision  is {gen_mean} -/+ {gen_std}.')
print(f'The number of particles analyzed in the dataset is {len(sigmas_list_df)}')

# **Create Medusa Plots**

In [ ]:
def calculate_pca_aspect(points):
    from numpy.linalg import eigh
    
    centered_points = points - points.mean(axis=0)
    cov = np.cov(centered_points, rowvar=False)
    eigvals, _ = eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    aspect = np.sqrt(eigvals[0]/ eigvals[1])
    return aspect

In [ ]:
from matplotlib.collections import LineCollection

aspects = []

for well, well_df in thresholded_data.groupby('Dataset'):
    print(well)
    fig, ax = plt.subplots(figsize = (10,8))
    plt.axis([-8, 8, -8, 8])
    points_all = []

    for _, traj_df in well_df.groupby('Trajectory'):
        traj = traj_df[['Time (s)', 'x (rel)', 'y (rel)']].to_numpy()
        print(traj)
        points = traj[:, 1:3]
        points_all.append(points)
        segments = np.stack([points[:-1], points[1:]], axis=1)

        t = traj[:,0]
        time_points = np.linspace(0, t[-1], len(t))

        lc = LineCollection(segments, cmap='jet', norm=plt.Normalize(0, t[-1]), alpha=1)

        lc.set_array(time_points[:len(segments)])
        lc.set_linewidth(1)
        ax.add_collection(lc)


    cbar=fig.colorbar(lc, ax=ax, label = 'Time (min)')
    #plt.title(f"Particles n={num_beads}", fontsize=20)
    cbar.set_label("Time (min)",fontsize=20)
    cbar.ax.tick_params(labelsize=16)

    points_all = np.vstack(points_all)


    plt.xlabel("x position (μm)", fontsize=32)
    plt.ylabel("y position (μm)", fontsize=32)
    plt.setp(ax.get_xticklabels(), fontsize=26)
    plt.setp(ax.get_yticklabels(), fontsize=26)

    plt.grid(True, linewidth=2)

    plt.show()

    aspect = calculate_pca_aspect(points_all)
    print(aspect)
    aspects.append([well, aspect])

for item in aspects: 
   print(f'{item[0]}: {item[1]:.2f}')

# **Plot Trajectories**

In [ ]:
for traj, traj_df in thresholded_data.groupby('Trajectory'):
    print(f'Plotting 1D traces for trajectory {traj}.')
    traj_array = [
        traj_df['FraTime (s)'].tolist(),
        traj_df['x (rel)'].tolist(),
        traj_df['y (rel)'].tolist()
    ]
    traj_array = np.column_stack(traj_array)

    plot_1D(traj_array)

# **Evaluate Step Length Distributions**

In [ ]:
steps_16 = []
steps_8 = []

for well, well_df in thresholded_data.groupby('Dataset'):
    print(well)
    if '16nM' in well:
        print('16')
    if 'RNase' in well:
        continue
    for _, traj_df in well_df.groupby('Trajectory'):
        traj = traj_df[['x (rel)', 'y (rel)']].to_numpy()
        x = traj_df['x (rel)'].tolist()
        y = traj_df['y (rel)'].tolist()
        steps_x = np.diff(x)
        steps_y = np.diff(y)
        mags = np.sqrt(steps_x**2 + steps_y**2) * 1000
        
        if '16nM' in well:
            steps_16.append(mags)

        else:
            steps_8.append(mags)

    
steps_16 = list(itertools.chain.from_iterable(steps_16))
steps_8 = list(itertools.chain.from_iterable(steps_8))
plt.figure(figsize = (10,8))
plt.ecdf(steps_16, complementary = True, label = '16 nM ExoIII')
plt.ecdf(steps_8, complementary = True, label = '8 nM ExoIII')
plt.xscale('log')
plt.yscale('log')
plt.show()

In [ ]:
# Boundary conditions selected to remove steps below 1sd of distributions for stalled particles calculated above, which are assumed to be indistinguishable from noise. 
# Upper bound is large enough that only a minute number of steps would be excluded, if at all, while simplifying comparisons between data sets. 
lower_bound = 41.35
upper_bound = 300
steps_16 = np.asarray(steps_16)
steps_8 = np.asarray(steps_8)


fig, ax = plt.subplots(figsize = (10,8))
ecdf_16 = ax.ecdf(steps_16, complementary = True, label = '16 nM ExoIII')
ecdf_8 = ax.ecdf(steps_8, complementary = True, label = '8 nM ExoIII')
ax.axvline(x=41, linestyle = 'dotted', color = 'k', label = r'$\sigma$')
#plt.axvline(x=82, color = 'k', linestyle = 'dotted', label = '2$\sigma$')
ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlabel('Distance (nm)', fontsize = 32)
ax.set_ylabel('P(s > d)', fontsize = 32)
ax.tick_params(axis = 'x', which = 'major', labelsize = 24)
ax.tick_params(axis = 'y', which = 'major', labelsize = 24)
ax.set_xlim(10, 2500)


x_16 = ecdf_16.get_xdata()[1:]
y_16 = ecdf_16.get_ydata()[1:]

x_8 = ecdf_8.get_xdata()[1:]
y_8 = ecdf_8.get_ydata()[1:]

mask_16 = (x_16 >= lower_bound) & (x_16 <= upper_bound)
mask_8 = (x_8 >= lower_bound) & (x_8 <= upper_bound)

print(f'{100*sum(mask_16)/len(steps_16):.2f} percent of values are between {lower_bound} and {upper_bound} nm for 16nM condition.')
print(f'{100*sum(mask_8)/len(steps_8):.2f} percent of values are between {lower_bound} and {upper_bound} nm for 8nM condition.')

x_16 = x_16[mask_16]
y_16 = y_16[mask_16]
m_16, c_16 = np.polyfit(np.log(x_16), np.log(y_16), 1)
y_pred_16 = np.exp(c_16) * x_16**m_16
idx_16 = np.argsort(x_16)
ax.plot(x_16[idx_16], y_pred_16[idx_16], color='k', linewidth=2, linestyle = 'dashed', label=rf'16 nM ExoIII fit, $\alpha$ = {m_16:.2f}')

x_8 = x_8[mask_8]
y_8 = y_8[mask_8]
m_8, c_8 = np.polyfit(np.log(x_8), np.log(y_8), 1)
y_pred_8 = np.exp(c_8) * x_8**m_8
idx_8 = np.argsort(x_8)
ax.plot(x_8[idx_8], y_pred_8[idx_8], color='k', linewidth=2, linestyle = 'dashdot', label=rf'16 nM ExoIII fit, $\alpha$ = {m_8:.2f}')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize = 16)
plt.show()

# **Determine % Active Motion**

Plan: Determine what fraction of particles have trajectories that are substantially misaligned with gravitational vector from stage tilt. Take steps with 95%CI not noise, then measure dot product with gravitational vector. Use unit gravitational vector, then calculate theta with theta = cos-1(a@b/mag(a)). < 30 degrees suggests active motion. Count fraction of trajectories with such a vector.



In [ ]:
def compute_step_lengths_angles(traj, primary_angle_rad, threshold):
    """
    Computes lengths of steps and angles between steps (movement vectors and angles between movement vectors).
    Angles are computed relative to USER DEFINED ANGLE, which is useful for an absolute comparison with stage angle. 
    The idea is that if the particle take a processive step against the stage tilt, where a processive step means above a 
    threshold determined by the localization accuracy, then it is actively moving. This is particularly salient for cases
    where frame rates are low.

    Parameters: 
    -   traj: np.ndarray
            Contains time series data for x and y coordinates. (T, D)
    
    -   primary_angle_rad: float
            Comparison angle in radians. 
    
    -   threshold: int or float
            All vectors below this value are assumed indistinguishable from noise and discarded. 

    Returns: 
    -   thetas: list
            List of vector angles relative to primary_angle_rad.
    """
    # Trajectory should be numpy array of dimensions (T, 2)
    vectors = [[b[0] - a[0], b[1] - a[1]] for a, b in zip(traj, traj[1:])]
    vectors = np.asarray(vectors)
    mags = np.array([np.linalg.norm(x) for x in vectors])

    primary_vec = [np.cos(primary_angle_rad), np.sin(primary_angle_rad)]

    thetas = []
    for i, mag in enumerate(mags): 
        if mag < threshold:
            continue
        vec = vectors[i]
        theta = np.arccos((vec @ primary_vec)/mag)
        thetas.append(theta)
    return thetas

In [ ]:
tilt_x_deg = -0.2
tilt_y_deg = -0.6

tilt_x = np.radians(tilt_x_deg)
tilt_y = np.radians(tilt_y_deg)

primary_angle_rad = np.arctan2(tilt_y, tilt_x)
print(primary_angle_rad)


plt.figure(figsize=(6,6))

# vector starting at origin
plt.quiver(0, 0, tilt_x, tilt_y, angles='xy', scale_units='xy', scale=1)

plt.xlim(-0.05,0.05)
plt.ylim(-0.05,0.05)
plt.axhline(0)
plt.axvline(0)

plt.xlabel("Tilt X")
plt.ylabel("Tilt Y")
plt.title("Tilt Vector")

plt.gca().set_aspect('equal')

plt.show()


    
angle_check_df = pd.DataFrame()

for well, well_df in tqdm.tqdm(thresholded_data.groupby('Dataset')):
    N = well_df['Trajectory'].nunique()

    arr = np.full(N, 0)
    i = 0
    for traj, traj_df in well_df.groupby('Trajectory'):
        active_check = 0
        thetas = compute_step_lengths_angles(traj_df[['x (rel)', 'y (rel)']].to_numpy(), primary_angle_rad=primary_angle_rad, threshold = 0.0414)
        thetas_mask = thetas <= np.sqrt(2)/2
        if np.sum(thetas_mask) > 1:
            active_check = 1
        assignment = active_check
        arr[i] = assignment
        i += 1
    print(fr"The percentage of active trajectories in well {well} is {100*np.sum(arr[:]) / len(arr)}")
    



In [ ]:
filepath = generate_output_filepath(file_string='Active.csv')

angle_check_df.to_csv(filepath, index=False)

# **Trajectory Simulation**

Build single-model theoretical datasets

In [ ]:
from andi_datasets.datasets_theory import datasets_theory

DT = datasets_theory()

# Converts integer function calls to corresponding string
motion_dict = {
    0 : 'attm',
    1 : 'ctrw',
    2 : 'fbm',
    3 : 'lw',
    4 : 'sbm'
}

# Simulation parameters
num_trajectories = 100
alpha_step = 0.01

# Initialize trajectory lists and id
subdiffusive_traj = []
superdiffusive_traj = []
i = 1

# Generate non-Levy motion dataset ------------------------------------------------------------------------------------
for key, value in motion_dict.items():

    # skip Levy walks
    if key == 3:
        continue

    print(f'Generating subdiffusive trajectories for {value} model...')

    dataset_subdiffusive = DT.create_dataset(
        T = 361,                                                            # trajectory length
        N_models = num_trajectories,                                        # number of trajectories per set of parameters
        exponents = np.arange(start = 0.5, stop = 1, step = alpha_step),    # anomalous diffusion coefficients
    models = [key],                                                         # list of model IDs to generate
        dimension = 2                                                       # Number of dimensions
    )

    for idx, row in tqdm.tqdm(enumerate(dataset_subdiffusive), desc = 'Compiling trajectories'):
        id = np.full(361, i)
        i += 1
        t = np.arange(1, 362)
        model = np.full(361, motion_dict[row[0]])
        a = np.full(361, row[1])
        x = row[2:363]
        y = row[363:]
        
        arr = np.column_stack([id, model, a, t, x, y])

        subdiffusive_traj.append(arr)

subdiffusive_arr = np.vstack(subdiffusive_traj)

df_subdiffusive = pd.DataFrame(subdiffusive_arr, columns = ['ID', 'Model', 'Alpha', 'Frame', 'x', 'y'])

# Generate Levy motion dataset (requires alpha > 1) ------------------------------------------------------------------------------------
for key, value in motion_dict.items():

    # skip attm and ctrw
    if key == 0 or key == 1:
        continue
    
    print(f'Generating superdiffusive trajectories for {value} model...')

    dataset_superdiffusive = DT.create_dataset(
        T = 361,                                                                # trajectory length
        N_models = num_trajectories,                                            # number of trajectories per set of parameters
        exponents = np.arange(start = 1.01, stop = 1.51, step = alpha_step),    # anomalous diffusion coefficients
        models = [key],                                                         # list of model IDs to generate
        dimension = 2                                                           # N dimensions
    )

    for idx, row in tqdm.tqdm(enumerate(dataset_superdiffusive), desc = 'Compiling trajectories'):
        id = np.full(361, i)
        i += 1
        t = np.arange(1, 362)
        model = np.full(361, motion_dict[row[0]])
        a = np.full(361, row[1])
        x = row[2:363]
        y = row[363:]
        
        arr = np.column_stack([id, model, a, t, x, y])

        superdiffusive_traj.append(arr)
    
superdiffusive_arr = np.vstack(superdiffusive_traj)

df_superdiffusive = pd.DataFrame(superdiffusive_arr, columns = ['ID', 'Model', 'Alpha', 'Frame', 'x', 'y'])

    
simulated_dataset = pd.concat([df_subdiffusive, df_superdiffusive])
simulated_dataset['ID'] = simulated_dataset['ID'].astype(int)
simulated_dataset['Frame'] = simulated_dataset['Frame'].astype(int)
simulated_dataset['Alpha'] = simulated_dataset['Alpha'].astype(float)
simulated_dataset['x'] = simulated_dataset['x'].astype(float)
simulated_dataset['y'] = simulated_dataset['y'].astype(float)
print(simulated_dataset['ID'].nunique())

print(simulated_dataset)

output_path_simdata = generate_output_filepath(file_string = 'Simulated_Dataset_Theoretical.csv')
simulated_dataset.to_csv(output_path_simdata, index = False)

Plot random sample of each motion type

In [ ]:
for model, model_df in simulated_dataset.groupby('Model'):
    ids = model_df['ID'].unique()
    ids = [int(x) for x in ids]
    trajs_low = int(ids[0])
    trajs_high = int(ids[-1])
    sel_ids = [random.randrange(trajs_low, trajs_high, step=1) for x in range(5)]
    for id in sel_ids:
        id_traj = model_df[model_df['ID'] == id][['Frame','x', 'y']].to_numpy()
        plot_2D(id_traj)

Single-State Phenomenological Simulation

In [ ]:
from andi_datasets.datasets_phenom import models_phenom
from scipy.stats import truncnorm

MP = models_phenom()


def single_state_dt(MP, N=1, T=200, Ds=[1], alphas=[1], L=None, dim=2, deltaT=1):

    trajs = []
    labels = []

    for _ in range(N):

        D = truncnorm.rvs(
            (0 - Ds[0]) / Ds[1], np.inf,
            loc=Ds[0],
            scale=Ds[1]
            )
        
        alpha = truncnorm.rvs(
            (0 - alphas[0]) / alphas[1],
            (2 - alphas[0]) / alphas[1],
            loc=alphas[0],
            scale=alphas[1]
            )

        traj, label = MP._single_state_traj(
            T=T,
            D=D,
            alpha=alpha,
            L=L,
            deltaT=deltaT,
            dim=dim
        )

        trajs.append(traj)
        labels.append(label)

    return np.array(trajs), np.array(labels)

# Simulation parameters
num_trajectories = 10000
num_frames = 361
boundary = 200
alphas = [1.25, 0.03] # alphas[1] is STANDARD DEVIATION FOR MODIFIED WRAPPER, *NOT* VARIANCE
Ds = [0.096, 5e-6] # Ds[1] is STANDARD DEVIATION FOR MODIFIED WRAPPER, *NOT* VARIANCE
deltaT = 5

print('Simulating Trajectories...')

trajs, labels = single_state_dt(
                    MP = MP,
                    N = num_trajectories,
                    T = num_frames,
                    alphas = alphas,
                    L = boundary,
                    Ds = Ds,
                    deltaT = deltaT,
                    dim = 2
                    )

print(labels.shape)

single_state_arr = []

for i in tqdm.tqdm(range(0,num_trajectories), desc = 'Compiling Simulated Trajectories'):
                    traj = trajs[i,:,:]
                    label = labels[i,:,:]
                    ids = np.full((num_frames, 1), i+1, dtype = int)
                    t = np.arange(1, num_frames+1).reshape(-1,1)
                    arr = np.hstack((ids, t, label[:, :2], traj))
                    single_state_arr.append(arr)

single_state_arr = np.vstack(single_state_arr)

ss_df = pd.DataFrame(single_state_arr, columns = ['ID', 'Frame', 'alpha', 'D', 'x', 'y'])
ss_df = ss_df.astype({'ID' : int, 'Frame' : int})

fig, ax = plt.subplots(1, 2, figsize = (6, 3), tight_layout = True)
ax[0].hist(labels[:,:,0].flatten(), bins = 50)
plt.setp(ax[0], title = r'Distribution of $\alpha$', xlabel = r'$\alpha$', ylabel = 'Frequency')

ax[1].hist(labels[:,:,1].flatten(), bins = 50)
plt.setp(ax[1], title = r'Distribution of $D$', xlabel = r'$D$', ylabel = 'Frequency')

output_path_ss_fig = generate_output_filepath(file_string = 'Simulated_Dataset_Phenomenological_Single-State_Params.png')
plt.savefig(output_path_ss_fig)

# for _, traj_df in ss_df.groupby('ID'):
#         print(fr"D = {traj_df['D'].iloc[0]:.2f}, $\alpha$ = {traj_df['alpha'].iloc[0]:.2f}")
#         plot_2D(traj_df[['Frame', 'x', 'y']].to_numpy(), real_time = True, frame_rate = 1/deltaT)


ids = ss_df['ID'].unique()
ids = [int(x) for x in ids]
trajs_low = int(ids[0])
trajs_high = int(ids[-1])
sel_ids = [random.randrange(trajs_low, trajs_high, step=1) for x in range(10)]
for id in sel_ids:
    id_traj = ss_df[ss_df['ID'] == id][['Frame','x', 'y']].to_numpy()
    plot_2D(id_traj, real_time = True)

output_path_ss_phenom = generate_output_filepath(file_string = 'Simulated_Dataset_Phenomenological_Single-State.csv')
ss_df.to_csv(output_path_ss_phenom, index = False)

Multi-State Phenomenological Simulation

In [ ]:
from andi_datasets.datasets_phenom import models_phenom
from scipy.stats import truncnorm

MP = models_phenom()


def two_state_dt(MP, M = [[0.95, 0.05], [0.05, 0.95]], N=1, T=200, Ds=[1], alphas=[1], L=None, dim=2, deltaT=1, return_state_num = True):

    trajs = []
    labels = []

    for _ in range(N):

        D0 = truncnorm.rvs(
            (0 - Ds[0][0]) / Ds[0][1], np.inf,
            loc=Ds[0][0],
            scale=Ds[0][1]
            )
        
        D1 = truncnorm.rvs(
            (0 - Ds[1][0]) / Ds[1][1], np.inf,
            loc=Ds[1][0],
            scale=Ds[1][1]
            )
        
        alpha0 = truncnorm.rvs(
            (0 - alphas[0][0]) / alphas[0][1],
            (2 - alphas[0][0]) / alphas[0][1],
            loc=alphas[0][0],
            scale=alphas[0][1]
            )
        
        alpha1 = truncnorm.rvs(
            (0 - alphas[1][0]) / alphas[1][1],
            (2 - alphas[1][0]) / alphas[1][1],
            loc=alphas[1][0],
            scale=alphas[1][1]
            )

        traj, label = MP._multiple_state_traj(
            T=T,
            Ds=[D0, D1],
            M = M,
            alphas=[alpha0, alpha1],
            L=L,
            deltaT=deltaT,
            return_state_num = return_state_num
        )

        trajs.append(traj)
        labels.append(label)

    return np.array(trajs), np.array(labels)

# Simulation parameters
num_trajectories = 10000
num_frames = 361
boundary = 200
transition_matrix = np.array([[0.7 , 0.3],[0.15, 0.85]])
Ds = np.array([[0.096, 2e-3], [0.01, 5e-4]])
alphas_ms = np.array([[1.3, 0.1], [0.5, 0.15]])

print('Simulating Trajectories...')

trajs, labels = two_state_dt(MP = MP,
                                N = num_trajectories,
                                T = num_frames,
                                L = boundary,
                                M = transition_matrix,
                                Ds = Ds,
                                alphas = alphas_ms,
                                #epsilon_a=[0.4], gamma_d = [0.75],
                                deltaT = 5,
                                return_state_num=True)



multi_state_arr = []

for i in tqdm.tqdm(range(0,num_trajectories), desc = 'Compiling Simulated Trajectories'):
                                traj = trajs[i,:,:]
                                label = labels[i,:,:]
                                ids = np.full((num_frames, 1), i+1, dtype = int)
                                t = np.arange(1, num_frames+1).reshape(-1,1)
                                arr = np.hstack((ids, t, label, traj))
                                multi_state_arr.append(arr)

multi_state_arr = np.vstack(multi_state_arr)

ms_df = pd.DataFrame(multi_state_arr, columns = ['ID', 'Frame', 'alpha', 'D', 'State Label','State', 'x', 'y'])
ms_df = ms_df.astype({'ID' : int, 'Frame' : int})

print(ms_df.head())

fig, ax = plt.subplots(1, 2, figsize = (6, 3), tight_layout = True)
ax[0].hist(labels[:,:,0].flatten(), bins = 50)
plt.setp(ax[0], title = r'Distribution of $\alpha$', xlabel = r'$\alpha$', ylabel = 'Frequency')

ax[1].hist(labels[:,:,1].flatten(), bins = 50)
plt.setp(ax[1], title = r'Distribution of $D$', xlabel = r'$D$', ylabel = 'Frequency')

output_path_ms_fig = generate_output_filepath(file_string = 'Simulated_Dataset_Phenomenological_Multi-State_Params.png')
plt.savefig(output_path_ms_fig)

# for _, traj_df in ms_df.groupby('ID'):
#     plot_2D(traj_df[['Frame', 'x', 'y']].to_numpy())

ids = ms_df['ID'].unique()
ids = [int(x) for x in ids]
trajs_low = int(ids[0])
trajs_high = int(ids[-1])
sel_ids = [random.randrange(trajs_low, trajs_high, step=1) for x in range(10)]
for id in sel_ids:
    id_traj = ms_df[ms_df['ID'] == id][['Frame','x', 'y']].to_numpy()
    plot_2D(id_traj, real_time=True)

output_path_ms_phenom = generate_output_filepath(file_string = 'Simulated_Dataset_Phenomenological_Multi-State.csv')
ms_df.to_csv(output_path_ms_phenom, index = False)

# **DeepSPT Benchmarking**

Read global fingerprint CSV files

In [ ]:
files = global_filepath.glob('*.csv')
files = sorted(files)
compiled_data = pd.DataFrame()

for file in files: 
    if 'Theoretical' in file.stem:
        continue
    df = pd.read_csv(file, index_col=False)
    df['Dataset'] = file.stem
    compiled_data = pd.concat([compiled_data, df])

print(compiled_data)
print(compiled_data['Dataset'].unique())


# **Principal Component Analysis**

Train PCA model

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

# Excludes columns that are either not needed, that have limited documentation in DeepSPT paper, or that are tautological (e.g. differentiating by diffusion coefficient when the diffusion coefficient is known for part of the dataset)
x = compiled_data.drop(['Particle ID', 'Dataset', 'Extra', 'D', 'meanSequence', 'medianSequence', 'maxSequence', 'minSequence', 'stdSequence', 'simSeq', 'logEfficiency', 'Alpha', 'inst_msd_D', 'pval'], axis = 1)
y = compiled_data['Dataset']

# Scale variables to avoid outsized impact of large scalar values
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

# Build PCA model
pca = PCA(n_components=3, svd_solver='full')
X_pca = pca.fit_transform(x_scaled)

X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

Plot PCA classification in confusion matrix

In [ ]:
tick_labels_list = ['16 nM ExoIII', '8 nM ExoIII', '5U RNase H', 'Simulated FBM', 'Simulated FBM with TAD']

cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(cm_norm, annot=True, cmap='Blues', xticklabels=tick_labels_list, yticklabels=tick_labels_list, cbar=True, vmin=0,
    vmax=1,
    cbar_kws={'label': 'Fraction of true class'})
ax.set_xlabel('Predicted Label', fontsize = 24)
ax.set_ylabel('True Label', fontsize = 24)
ax.tick_params(which = 'both', labelsize = 16)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
cbar = ax.collections[0].colorbar
cbar.set_label('Fraction of true class', fontsize=20)
cbar.ax.tick_params(labelsize=16)
fig.suptitle('Confusion Matrix', fontsize = 24)
plt.show()

Plot populations with respect to first two principal components

In [ ]:
from matplotlib.colors import BoundaryNorm

y_numeric = pd.factorize(y)[0]
print(y_numeric)
print(y)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(x_scaled[:, 0], x_scaled[:, 1], c=y_numeric, cmap='coolwarm', edgecolor='k', s=8)
plt.xlabel('Original Feature 1')
plt.ylabel('Original Feature 2')
plt.title('Before PCA: Using First 2 Standardized Features')
plt.colorbar(label='Target classes')

plt.subplot(1, 2, 2)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_numeric, cmap='coolwarm', edgecolor='k', s=8)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title(f'After PCA: Projected onto {pca.n_components_} Principal Components')
plt.colorbar(label='Target classes')

plt.tight_layout()
plt.show()


classes = np.unique(y_numeric)
n_classes = len(classes)

cmap = plt.get_cmap('coolwarm', n_classes)
norm = BoundaryNorm(np.arange(-0.5, n_classes + 0.5, 1), n_classes)

fig, ax = plt.subplots(figsize = (10,8))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y_numeric, cmap='coolwarm', edgecolor='k', s=8)
ax.set_xlabel('Principal Component 1', fontsize = 24)
ax.set_ylabel('Principal Component 2', fontsize = 24)
fig.suptitle(f'After PCA: Projected onto {pca.n_components_} Principal Components', fontsize = 24)
cbar = plt.colorbar(sc, ticks=classes)
cbar.set_label('Target Classes', fontsize = 24)
cbar.ax.tick_params(labelsize=18)
ax.tick_params(labelsize = 18)
plt.show()

feature_names = x.columns
loadings = pd.DataFrame(
    pca.components_,
    columns=feature_names,
    index=[f'PC{i+1}' for i in range(pca.n_components_)]
)

loadings['Variance'] = pca.explained_variance_ratio_
print(loadings)
# loadings_path = generate_output_filepath(file_string='Motor PCA Loadings_Empirical-Theoretical.csv')
# loadings.to_csv(loadings_path)

loadings['Eigen'] = pca.explained_variance_

fig, ax = plt.subplots()
ax.plot(loadings['Eigen'])
plt.show()

In [ ]:
from matplotlib.colors import BoundaryNorm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import LabelEncoder

class_map = {'16 nM ExoIII': 0, '8 nM ExoIII': 1, '5U RNase H': 2, 'Single-State FBM': 3, 'FBM with TAD' : 4, 'FBM' : 5, 'Levy Walk' : 6, 'SBM' : 7}
inv_class_map = {v: k for k, v in class_map.items()}
class_names = [inv_class_map[i] for i in classes]

# Numeric class IDs used for coloring
classes = np.unique(y_numeric)
n_classes = len(classes)

# If you used LabelEncoder:
#class_names = le.classes_   # <-- string labels in same order as numeric codes

# Discrete colormap + discrete normalization
cmap = plt.get_cmap('coolwarm', n_classes)
norm = BoundaryNorm(np.arange(-0.5, n_classes + 0.5, 1), n_classes)

fig, ax = plt.subplots(figsize=(10, 8))

sc = ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=y_numeric,
    cmap=cmap,
    norm=norm,
    edgecolor='k',
    s=8
)

ax.set_xlabel('Principal Component 1', fontsize=24)
ax.set_ylabel('Principal Component 2', fontsize=24)
fig.suptitle(f'After PCA: Projected onto {pca.n_components_} Principal Components', fontsize=24)

# Colorbar ticks at numeric class IDs, labels replaced with class names
cbar = plt.colorbar(sc, ticks=classes)
cbar.set_ticklabels(class_names)
cbar.set_label('Target Classes', fontsize=24)
cbar.ax.tick_params(labelsize=18)

ax.tick_params(labelsize=18)
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))

for class_id, class_name in reversed(list(enumerate(class_names))):
    mask = y_numeric == class_id
    ax.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        label=class_name,
        s=12,
        edgecolor='k',
        alpha=0.8
    )

ax.set_xlabel('Principal Component 1', fontsize=24)
ax.set_ylabel('Principal Component 2', fontsize=24)
#ax.set_title(f'Principal Component Analysis of Empirical Rolling Motors and Simulated FBM Trajectories', fontsize=24)
ax.tick_params(labelsize=18)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], fontsize=14, markerscale=1.5, title_fontsize=16)

plt.show()

Additional functions

In [ ]:
import numpy as np
import pandas as pd

# Map numeric labels to class names
y_string = np.array([inv_class_map[i] for i in y_numeric])  # shape (n_samples,)

# Make it a column vector
y_col = y_string[:, np.newaxis]  # shape (n_samples, 1)

# Horizontally stack PCA components with class column
output_array = np.hstack((X_pca, y_col))  # shape (n_samples, n_features + 1)

# Generate column names
pc_columns = [f'PC{i+1}' for i in range(X_pca.shape[1])]
columns = pc_columns + ['Class']

# Create DataFrame
df_pca = pd.DataFrame(output_array, columns=columns)

# Optional: save to CSV
output_path = generate_output_filepath(file_string = 'PCA_Output-3Components.csv')
df_pca.to_csv(output_path, index=False)

print(df_pca.head())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import ConvexHull

fig, ax = plt.subplots(figsize=(10, 8))

for class_id, class_name in reversed(list(enumerate(class_names))):
    # Mask points for this class
    mask = y_numeric == class_id
    points = X_pca[mask, :2]  # first 2 PCs

    if len(points) < 3:
        # ConvexHull requires at least 3 points
        continue

    # Compute convex hull
    hull = ConvexHull(points)

    # Plot the hull outline (closed loop)
    hull_points = points[hull.vertices]
    
    if 'FBM' in class_name:
        linestyle = '--'
    else:
        linestyle = '-'
    
    ax.plot(
        np.append(hull_points[:, 0], hull_points[0, 0]),
        np.append(hull_points[:, 1], hull_points[0, 1]),
        label=class_name,
        linewidth=2,
        linestyle = linestyle
    )

ax.set_xlabel('Principal Component 1', fontsize=24)
ax.set_ylabel('Principal Component 2', fontsize=24)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], fontsize=14, markerscale=1.5, title_fontsize=16)
ax.tick_params(labelsize=18)
plt.show()